# Notebook 1 — Trích xuất Embeddings (.npy)

Output: `item_visual_features.npy`, `item_text_features.npy`, `item_embeddings.npy`, `user_embeddings.npy`, `outfit_embeddings.npy`

**Lọc dữ liệu theo active-user pipeline (MIN_USER_INTERACTIONS ≥ 4)**

## 0. Cài đặt

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}')
print('✅ Install OK — hyperparameters in fgat_config.py')


PyTorch: 2.12.0
CUDA: False | MPS: True
✅ Install OK — hyperparameters in fgat_config.py


## 1. Imports & Setup

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, random, requests
from PIL import Image
from torchvision import models, transforms
from transformers import BertTokenizer, BertModel
import fgat_config as cfg

SEED = cfg.RANDOM_SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device} (type={device.type})')


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps (type=mps)


In [3]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import fgat_config as cfg
importlib.reload(cfg)  # pick up edits to fgat_config.py without kernel restart
from fgat_config import resolve_paths, print_config_summary

_paths = resolve_paths(REPO_ROOT)
DATA_ROOT   = _paths['DATA_ROOT']
OUTPUT_ROOT = _paths['OUTPUT_ROOT']
IMAGE_ROOT  = _paths['IMAGE_ROOT']
CACHE_DIR   = _paths['CACHE_DIR']
DATA_PATH   = _paths['DATA_PATH']
OUTPUT_PATH = _paths['OUTPUT_PATH']
IMAGE_DIR   = _paths['IMAGE_DIR']
FEATURE_CACHE_PATH = _paths['FEATURE_CACHE_PATH']

MIN_USER_INTERACTIONS = cfg.MIN_USER_INTERACTIONS
FORCE_REBUILD_FEATURES = cfg.FORCE_REBUILD_FEATURES
IMAGE_BATCH_SIZE = cfg.IMAGE_BATCH_SIZE
TEXT_BATCH_SIZE = cfg.TEXT_BATCH_SIZE
MAX_TEXT_LENGTH = cfg.MAX_TEXT_LENGTH
EMBED_DIM = cfg.EMBED_DIM
VISUAL_DIM = cfg.VISUAL_DIM
TEXT_DIM = cfg.TEXT_DIM

def file_exists(path):
    exists = os.path.exists(path)
    status = '✅ Đã có' if exists else '⏳ Chưa có'
    print(f'  {status}: {os.path.basename(path)}')
    return exists

def parse_ids(s):
    s = str(s).strip()
    sep = ';' if ';' in s else ' '
    return [int(x.strip()) for x in s.split(sep) if x.strip()]

print_config_summary()
print(f'  DATA_PATH   = {DATA_PATH}')
print(f'  OUTPUT_PATH = {OUTPUT_PATH}')
print(f'  IMAGE_DIR   = {IMAGE_DIR}')
print(f'  CACHE       = {FEATURE_CACHE_PATH}')
print('✅ Paths & config OK')


── fgat_config ──
  MIN_USER_INTERACTIONS=4  SPLIT_MODE=per_user
  MIN_TOP_NEIGHBORS=None  (None=disabled)
  EPOCHS=50  LR=0.001  WEIGHT_DECAY=1e-05  LAMBDA_COMP=0.3
  COMPAT_DETACH_INPUT=True  COMPAT_BPR_MARGIN=0.0  COMPAT_LR_MULT=1.0
  BATCH_SIZE=512  NEG_PER_POS=3  PATIENCE=10
  SCHEDULER_PATIENCE=5  LEARNABLE_EMBEDDINGS=True
  EVAL_EVERY=1  EARLY_STOP_METRIC=HR@K
  IMAGE_BATCH_SIZE=64  TEXT_BATCH_SIZE=128
  MAX_TEXT_LENGTH=ad-hoc (BERT max)  FORCE_REBUILD_FEATURES=False
  DATA_PATH   = /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/hfgat_rewrite_validate/Dataset/
  OUTPUT_PATH = /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/
  IMAGE_DIR   = /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/hfgat_rewrite_validate/Dataset/fashion_item_images/
  CACHE      

## Stage 1A — Load dữ liệu thô

In [4]:
print('=== STAGE 1: Load dữ liệu thô ===')

item_data   = pd.read_csv(DATA_PATH + 'item_data.txt',   header=None,
                          names=['item_id', 'category', 'image_url', 'title'])
outfit_data = pd.read_csv(DATA_PATH + 'outfit_data.txt', header=None,
                          names=['outfit_id', 'items'])
user_data   = pd.read_csv(DATA_PATH + 'user_data.txt',   header=None,
                          names=['user_id', 'outfits'])

# Ép kiểu
item_data['item_id']     = item_data['item_id'].astype(int)
outfit_data['outfit_id'] = outfit_data['outfit_id'].astype(int)
user_data['user_id']     = user_data['user_id'].astype(int)

# Load train_uo.txt nếu có (ưu tiên dùng file này để build edges)
TRAIN_UO_PATH = DATA_PATH + 'train_uo.txt'
if os.path.exists(TRAIN_UO_PATH):
    edges = []
    with open(TRAIN_UO_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2: continue
            user = int(parts[0])
            for o in parts[1:]:
                edges.append((user, int(o)))
    train_uo = pd.DataFrame(edges, columns=['user_id', 'outfit_id'])
    print(f'  train_uo.txt     : {len(train_uo):,} edges (từ file)')
else:
    # Suy ra từ user_data['outfits']
    edges = []
    for _, row in user_data.iterrows():
        u = int(row['user_id'])
        for o in parse_ids(row['outfits']):
            edges.append((u, o))
    train_uo = pd.DataFrame(edges, columns=['user_id', 'outfit_id'])
    print(f'  train_uo (derived): {len(train_uo):,} edges (suy ra từ user_data)')

print(f'  Items raw   : {len(item_data):,} | Categories: {item_data["category"].nunique()}')
print(f'  Outfits raw : {len(outfit_data):,}')
print(f'  Users raw   : {user_data["user_id"].nunique():,}')
print('✅ Stage 1 hoàn thành!')

=== STAGE 1: Load dữ liệu thô ===
  train_uo (derived): 679,028 edges (suy ra từ user_data)
  Items raw   : 19,175 | Categories: 61
  Outfits raw : 9,373
  Users raw   : 277,469
✅ Stage 1 hoàn thành!


## Stage 1B — Download ảnh

In [5]:

#****************Source gốc tải ảnh bị quá chậm --> custom phần download

import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import local

file_path = os.path.join(DATA_PATH, "item_data.txt")

item_data = pd.read_csv(
    file_path, header=None,
    names=['item_id', 'category', 'image_url', 'title']
)

image_dir = os.path.join(OUTPUT_PATH, "images")
os.makedirs(image_dir, exist_ok=True)

# Resume
existing_files = set(
    f.split(".")[0] for f in os.listdir(image_dir) if f.endswith(".png")
)

item_data_sample = item_data[
    ~item_data["item_id"].astype(str).isin(existing_files)
]

total_images = len(item_data_sample)
print(f"🟢 Đã có {len(existing_files)} ảnh | 🟡 Còn {total_images} ảnh cần tải")

headers = {"User-Agent": "Mozilla/5.0"}

_thread_local = local()

def get_session():
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.headers.update(headers)
        _thread_local.session = s
    return _thread_local.session


def download_image(image_url, image_id, retries=2):
    image_path = os.path.join(image_dir, f"{image_id}.png")

    if os.path.exists(image_path):
        return image_id, image_path

    session = get_session()

    for _ in range(retries):
        try:
            r = session.get(image_url, stream=True, timeout=5)
            if r.status_code == 200:
                with open(image_path, "wb") as f:
                    for chunk in r.iter_content(1024):
                        if chunk:
                            f.write(chunk)
                return image_id, image_path
        except requests.exceptions.RequestException:
            continue

    return image_id, None


max_threads = 15
image_paths = {}

completed = 0

with ThreadPoolExecutor(max_workers=max_threads) as ex:
    futures = [
        ex.submit(download_image, row.image_url, row.item_id)
        for row in item_data_sample.itertuples(index=False)
    ]

    for future in as_completed(futures):
        completed += 1
        item_id, path = future.result()

        if path:
            image_paths[item_id] = path
            status = "✅"
        else:
            status = "❌"

        print(f"{status} {completed}/{total_images} | item_id: {item_id}")

print("🎉 Download hoàn tất!")


🟢 Đã có 19167 ảnh | 🟡 Còn 8 ảnh cần tải
❌ 1/8 | item_id: 18133
❌ 2/8 | item_id: 9323
❌ 3/8 | item_id: 12552
❌ 4/8 | item_id: 7362
❌ 5/8 | item_id: 14835
❌ 6/8 | item_id: 18749
❌ 7/8 | item_id: 16246
❌ 8/8 | item_id: 11601
🎉 Download hoàn tất!


## Stage 1C — Active-user filtering pipeline

Lọc theo chuỗi nhân quả:
1. Dedup edges (user, outfit)
2. Chỉ giữ user có ≥ MIN_USER_INTERACTIONS
3. Lấy outfit xuất hiện trong tập user đã lọc
4. Lấy item từ các outfit đó
5. Chỉ giữ item có ảnh thật trên disk
6. Lọc lại outfit: bỏ item không có ảnh, bỏ outfit trống
7. Lọc lại edges theo outfit cuối
8. Lọc lại user theo edges cuối
9. Lọc lại item theo outfit cuối

In [6]:
print('=== STAGE 1C: Active-user filtering pipeline ===')
print(f'  Ngưỡng MIN_USER_INTERACTIONS = {MIN_USER_INTERACTIONS}')

# ── Bước 1: Dedup edges ───────────────────────────────────────────────
train_uo_base = train_uo.drop_duplicates(subset=['user_id', 'outfit_id']).copy().reset_index(drop=True)
print(f'\n  [1] Sau dedup edges: {len(train_uo_base):,} edges')

# ── Bước 2: Chỉ giữ user có >= MIN_USER_INTERACTIONS ─────────────────
user_inter_counts = train_uo_base['user_id'].value_counts()
active_users      = set(user_inter_counts[user_inter_counts >= MIN_USER_INTERACTIONS].index.tolist())
train_uo_sub      = train_uo_base[train_uo_base['user_id'].isin(active_users)].copy().reset_index(drop=True)

print(f'  [2] Users có >= {MIN_USER_INTERACTIONS} interactions: {len(active_users):,}')
print(f'      Edges còn lại: {len(train_uo_sub):,}')
print(f'      Avg interactions/user: {len(train_uo_sub)/max(1,len(active_users)):.1f}')

# ── Bước 3: Lấy outfit xuất hiện trong tập user đã lọc ───────────────
valid_outfit_ids = set(train_uo_sub['outfit_id'].unique().tolist())
outfit_sub       = outfit_data[outfit_data['outfit_id'].isin(valid_outfit_ids)].copy().reset_index(drop=True)
print(f'  [3] Outfits liên quan đến active users: {len(outfit_sub):,}')

# ── Bước 4: Lấy item từ các outfit đó ────────────────────────────────
valid_item_ids = set()
for items_str in outfit_sub['items']:
    valid_item_ids.update(parse_ids(items_str))
item_sub = item_data[item_data['item_id'].isin(valid_item_ids)].copy().reset_index(drop=True)
print(f'  [4] Items thuộc các outfit đó: {len(item_sub):,}')

# ── Bước 5: Chỉ giữ item có ảnh thật trên disk ───────────────────────
def find_image_path(item_id, image_dir):
    for ext in ['.png', '.jpg', '.jpeg', '.webp']:
        p = os.path.join(image_dir, f'{item_id}{ext}')
        if os.path.exists(p):
            return p
    return None

item_sub['image_path'] = item_sub['item_id'].apply(lambda x: find_image_path(x, IMAGE_DIR))
item_sub['has_image']  = item_sub['image_path'].notnull()
n_before = len(item_sub)
item_sub  = item_sub[item_sub['has_image']].copy().reset_index(drop=True)
print(f'  [5] Items có ảnh trên disk: {len(item_sub):,} (loại {n_before - len(item_sub):,} item thiếu ảnh)')

final_item_ids = set(item_sub['item_id'].tolist())

# ── Bước 6: Lọc lại outfit: bỏ item thiếu ảnh, bỏ outfit trống ────────
def filter_outfit_items(items_str):
    items = [x for x in parse_ids(items_str) if x in final_item_ids]
    return ';'.join(map(str, items))

outfit_sub['items'] = outfit_sub['items'].apply(filter_outfit_items)
n_before = len(outfit_sub)
outfit_sub = outfit_sub[outfit_sub['items'].apply(len) > 0].copy().reset_index(drop=True)
print(f'  [6] Outfits sau khi bỏ item thiếu ảnh: {len(outfit_sub):,} (loại {n_before - len(outfit_sub):,})')

# ── Bước 7: Lọc lại edges theo outfit cuối ────────────────────────────
final_outfit_ids = set(outfit_sub['outfit_id'].tolist())
train_uo_sub = train_uo_sub[train_uo_sub['outfit_id'].isin(final_outfit_ids)].copy().reset_index(drop=True)
print(f'  [7] Edges sau khi lọc outfit cuối: {len(train_uo_sub):,}')

# ── Bước 8: Lọc lại user theo outfit cuối ──────────────────────────────
final_user_ids = set(train_uo_sub["user_id"].tolist())
user_sub = user_data[user_data["user_id"].isin(final_user_ids)].copy().reset_index(drop=True)
print(f'  [8] Filtered users: {train_uo_sub["user_id"].nunique():,}')

# ── Bước 9: Lọc lại item theo outfit cuối ─────────────────────────────
used_item_ids = set()
for items_str in outfit_sub['items']:
    used_item_ids.update(parse_ids(items_str))
item_sub = item_sub[item_sub['item_id'].isin(used_item_ids)].copy().reset_index(drop=True)
print(f'  [9] Items cuối (chỉ item trong outfit cuối): {len(item_sub):,}')

# ── Tóm tắt ───────────────────────────────────────────────────────────
print(f'\n✅ Sau filter:')
print(f'  Items   : {len(item_sub):,}  (có ảnh & thuộc outfit)')
print(f'  Outfits : {len(outfit_sub):,}')
print(f'  Users   : {train_uo_sub["user_id"].nunique():,}  (≥ {MIN_USER_INTERACTIONS} interactions)')
print(f'  Edges   : {len(train_uo_sub):,}')
print(f'  Avg interactions/user: {len(train_uo_sub)/max(1,len(user_sub)):.1f}')

# ── Sanity check ──────────────────────────────────────────────────────
edge_outfit_ids = set(train_uo_sub['outfit_id'].unique())
assert edge_outfit_ids <= final_outfit_ids, 'FAIL: edge có outfit không tồn tại trong outfit_sub!'
outfit_item_ids = set()
for s in outfit_sub['items']: outfit_item_ids.update(parse_ids(s))
assert outfit_item_ids <= used_item_ids, 'FAIL: outfit có item không tồn tại trong item_sub!'
print('  Sanity check: ✅ tất cả IDs nhất quán')

# ── Gán biến toàn cục cho các stage sau ───────────────────────────────
item_data_filtered   = item_sub.copy()
outfit_data_filtered = outfit_sub.copy()
user_data_filtered   = user_sub.copy()
train_uo_filtered    = train_uo_sub.copy()

=== STAGE 1C: Active-user filtering pipeline ===
  Ngưỡng MIN_USER_INTERACTIONS = 4

  [1] Sau dedup edges: 679,028 edges
  [2] Users có >= 4 interactions: 25,263
      Edges còn lại: 127,536
      Avg interactions/user: 5.0
  [3] Outfits liên quan đến active users: 6,622
  [4] Items thuộc các outfit đó: 14,423
  [5] Items có ảnh trên disk: 14,419 (loại 4 item thiếu ảnh)
  [6] Outfits sau khi bỏ item thiếu ảnh: 6,622 (loại 0)
  [7] Edges sau khi lọc outfit cuối: 127,536
  [8] Filtered users: 25,263
  [9] Items cuối (chỉ item trong outfit cuối): 14,419

✅ Sau filter:
  Items   : 14,419  (có ảnh & thuộc outfit)
  Outfits : 6,622
  Users   : 25,263  (≥ 4 interactions)
  Edges   : 127,536
  Avg interactions/user: 1.0
  Sanity check: ✅ tất cả IDs nhất quán


## Stage 1D — Lưu subsample artifacts

In [7]:
print('=== STAGE 1D: Lưu subsample artifacts ===')

item_data_filtered[['item_id', 'category', 'image_url', 'title']].to_csv(
    OUTPUT_PATH + 'subsample/item_sub.csv', index=False)
outfit_data_filtered[['outfit_id', 'items']].to_csv(
    OUTPUT_PATH + 'subsample/outfit_sub.csv', index=False)
user_data_filtered[['user_id', 'outfits']].to_csv(
    OUTPUT_PATH + 'subsample/user_sub.csv', index=False)
train_uo_filtered.to_csv(
    OUTPUT_PATH + 'subsample/train_uo_sub.csv', index=False)

import json
with open(OUTPUT_PATH + 'subsample/filter_stats.json', 'w') as f:
    json.dump({
        'min_user_interactions': MIN_USER_INTERACTIONS,
        'items': int(len(item_data_filtered)),
        'outfits': int(len(outfit_data_filtered)),
        'users': int(len(user_data_filtered)),
        'edges': int(len(train_uo_filtered)),
    }, f, indent=2)

print('  💾 Đã lưu subsample/ artifacts')
print('✅ Stage 1C hoàn thành!')

=== STAGE 1D: Lưu subsample artifacts ===
  💾 Đã lưu subsample/ artifacts
✅ Stage 1C hoàn thành!


## Stage 2A — Batched features + cache (`item_features.pt`)


In [8]:
print('=== STAGE 2A: Batched feature extraction + cache ===')

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

item_ids_ordered = sorted(item_data_filtered['item_id'].tolist())
categories = sorted(item_data_filtered['category'].astype(str).unique().tolist())
cat2idx = {c: i for i, c in enumerate(categories)}
n_items = len(item_ids_ordered)
id_to_row = {row['item_id']: row for _, row in item_data_filtered.iterrows()}

if FEATURE_CACHE_PATH.exists() and not FORCE_REBUILD_FEATURES:
    cached = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
    image_feat_t = cached['image_feat'].float()
    text_feat_t  = cached['text_feat'].float()
    cat_feat_t   = cached['cat_feat'].float()
    item_ids_cached = cached['item_ids']
    assert list(item_ids_cached) == item_ids_ordered, 'Cache item order mismatch — set FORCE_REBUILD_FEATURES=True'
    print(f'  ✅ Loaded cache: {FEATURE_CACHE_PATH}')
else:
    print('  ⏳ Extracting features (batched)...')
    weights = models.ResNet152_Weights.DEFAULT if cfg.USE_PRETRAINED_BACKBONES else None
    resnet = models.resnet152(weights=weights)
    resnet = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
    if weights is not None:
        transform = weights.transforms()
    else:
        transform = transforms.Compose([
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    tokenizer = BertTokenizer.from_pretrained(cfg.TEXT_MODEL_NAME)
    bert_model = BertModel.from_pretrained(cfg.TEXT_MODEL_NAME).to(device).eval()

    class _ImgDS(Dataset):
        def __init__(self, ids, rows, tfm):
            self.ids, self.rows, self.tfm = ids, rows, tfm
        def __len__(self): return len(self.ids)
        def __getitem__(self, i):
            iid = self.ids[i]
            path = self.rows[iid]['image_path']
            try:
                img = Image.open(path).convert('RGB')
                return iid, self.tfm(img)
            except Exception:
                return iid, torch.zeros(3, 224, 224)

    img_loader = DataLoader(_ImgDS(item_ids_ordered, id_to_row, transform),
                            batch_size=IMAGE_BATCH_SIZE, shuffle=False, num_workers=0)
    image_rows = []
    with torch.no_grad():
        for batch_ids, batch_imgs in img_loader:
            feats = resnet(batch_imgs.to(device)).squeeze(-1).squeeze(-1).cpu()
            for iid, feat in zip(batch_ids.tolist(), feats):
                image_rows.append((iid, feat.numpy().astype(np.float32)))
    image_rows.sort(key=lambda x: item_ids_ordered.index(x[0]))
    image_feat_t = torch.tensor(np.stack([f for _, f in image_rows]), dtype=torch.float32)

    titles = [str(id_to_row[iid]['title']) if pd.notna(id_to_row[iid]['title']) else '' for iid in item_ids_ordered]
    text_chunks = []
    with torch.no_grad():
        for start in range(0, len(titles), TEXT_BATCH_SIZE):
            batch_text = titles[start:start + TEXT_BATCH_SIZE]
            tok_kwargs = dict(padding=True, truncation=True, return_tensors='pt')
            if MAX_TEXT_LENGTH is not None:
                tok_kwargs['max_length'] = MAX_TEXT_LENGTH
            enc = tokenizer(batch_text, **tok_kwargs)
            enc = {k: v.to(device) for k, v in enc.items()}
            cls = bert_model(**enc).last_hidden_state[:, 0, :].cpu()
            text_chunks.append(cls)
    text_feat_t = torch.cat(text_chunks, dim=0).float()

    cat_feat_t = torch.zeros(n_items, len(categories), dtype=torch.float32)
    for i, iid in enumerate(item_ids_ordered):
        c = str(id_to_row[iid]['category'])
        cat_feat_t[i, cat2idx[c]] = 1.0

    torch.save({
        'image_feat': image_feat_t,
        'text_feat': text_feat_t,
        'cat_feat': cat_feat_t,
        'item_ids': item_ids_ordered,
        'cat2idx': cat2idx,
    }, FEATURE_CACHE_PATH)
    print(f'  💾 Saved cache → {FEATURE_CACHE_PATH}')

item_visual_features = {iid: image_feat_t[i].numpy() for i, iid in enumerate(item_ids_ordered)}
item_text_features   = {iid: text_feat_t[i].numpy() for i, iid in enumerate(item_ids_ordered)}
print(f'  image_feat: {tuple(image_feat_t.shape)}  text_feat: {tuple(text_feat_t.shape)}  cat_feat: {tuple(cat_feat_t.shape)}')
print('✅ Stage 2A hoàn thành!')


=== STAGE 2A: Batched feature extraction + cache ===
  ✅ Loaded cache: /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/cache/item_features.pt
  image_feat: (14419, 2048)  text_feat: (14419, 768)  cat_feat: (14419, 59)
✅ Stage 2A hoàn thành!


## Stage 2B — (merged into 2A)


In [9]:
print('=== STAGE 2B: (merged into 2A) ===')
print('  Text/visual/category features are in item_features.pt')
print('✅ Stage 2B skipped')


=== STAGE 2B: (merged into 2A) ===
  Text/visual/category features are in item_features.pt
✅ Stage 2B skipped


## Stage 2C — Fuse visual + text → item embedding


In [10]:
print('=== STAGE 2C: Kết hợp Visual + Text → Item Embedding ===')

ITEM_EMB_FILE = OUTPUT_PATH + 'embeddings/item_embeddings.npy'
D_M = EMBED_DIM

if file_exists(ITEM_EMB_FILE):
    item_embeddings_arr = np.load(ITEM_EMB_FILE)
    print(f'  Đã load item_embeddings shape: {item_embeddings_arr.shape}')
else:
    class ItemEmbedding(nn.Module):
        def __init__(self, visual_dim=VISUAL_DIM, text_dim=TEXT_DIM, output_dim=EMBED_DIM):
            super().__init__()
            self.fc = nn.Linear(visual_dim + text_dim, output_dim)
        def forward(self, visual_feat, text_feat):
            combined = torch.cat([
                torch.tensor(visual_feat, dtype=torch.float32),
                torch.tensor(text_feat, dtype=torch.float32),
            ], dim=0)
            return self.fc(combined).detach().numpy()

    emb_model = ItemEmbedding()
    embeddings_list = []
    for item_id in item_ids_ordered:
        v_feat = item_visual_features.get(item_id, np.zeros(VISUAL_DIM, dtype=np.float32))
        t_feat = item_text_features.get(item_id, np.zeros(TEXT_DIM, dtype=np.float32))
        emb = emb_model(v_feat, t_feat)
        embeddings_list.append([item_id] + emb.tolist())

    item_embeddings_arr = np.array(embeddings_list, dtype=np.float32)
    np.save(ITEM_EMB_FILE, item_embeddings_arr)
    pd.DataFrame(item_embeddings_arr, columns=['item_id'] + [f'dim_{i}' for i in range(D_M)]).to_csv(
        OUTPUT_PATH + 'embeddings/item_embeddings.csv', index=False)
    print(f'  💾 Đã lưu: {ITEM_EMB_FILE}  shape={item_embeddings_arr.shape}')

print(f'✅ Stage 2C hoàn thành! shape={item_embeddings_arr.shape}')


=== STAGE 2C: Kết hợp Visual + Text → Item Embedding ===
  ✅ Đã có: item_embeddings.npy
  Đã load item_embeddings shape: (14419, 65)
✅ Stage 2C hoàn thành! shape=(14419, 65)


## Stage 2D — User & Outfit Embedding

In [11]:
print('=== STAGE 2D: Khởi tạo User & Outfit Embedding ===')

USER_EMB_FILE   = OUTPUT_PATH + 'embeddings/user_embeddings.npy'
OUTFIT_EMB_FILE = OUTPUT_PATH + 'embeddings/outfit_embeddings.npy'
D_M = EMBED_DIM

def save_random_embeddings(entity_ids, file_path, entity_name):
    if file_exists(file_path):
        arr = np.load(file_path)
        print(f'  Đã load {entity_name}_embeddings shape: {arr.shape}')
        return arr
    embeddings_list = [[eid] + np.random.uniform(-0.1, 0.1, D_M).astype(np.float32).tolist()
                       for eid in entity_ids]
    arr = np.array(embeddings_list, dtype=np.float32)
    np.save(file_path, arr)
    pd.DataFrame(arr, columns=['id'] + [f'dim_{i}' for i in range(D_M)]).to_csv(
        file_path.replace('.npy', '.csv'), index=False)
    print(f'  💾 Đã lưu: {file_path}  shape={arr.shape}')
    return arr

user_ids_list   = sorted(user_data_filtered['user_id'].unique().tolist())
outfit_ids_list = sorted(outfit_data_filtered['outfit_id'].tolist())
user_embeddings_arr   = save_random_embeddings(user_ids_list, USER_EMB_FILE, 'user')
outfit_embeddings_arr = save_random_embeddings(outfit_ids_list, OUTFIT_EMB_FILE, 'outfit')
print('✅ Stage 2D hoàn thành!')


=== STAGE 2D: Khởi tạo User & Outfit Embedding ===
  ✅ Đã có: user_embeddings.npy
  Đã load user_embeddings shape: (25263, 65)
  ✅ Đã có: outfit_embeddings.npy
  Đã load outfit_embeddings shape: (6622, 65)
✅ Stage 2D hoàn thành!


## Stage 3 — Tóm tắt toàn bộ

In [12]:
print('=== TỔNG KẾT Notebook 1 ===')
print(f'  MIN_USER_INTERACTIONS : {MIN_USER_INTERACTIONS}')
print(f'')
print(f'  Items sau filter  : {len(item_data_filtered):,}  (có ảnh, thuộc outfit)')
print(f'  Outfits sau filter: {len(outfit_data_filtered):,}')
print(f'  Users sau filter  : {len(user_data_filtered):,}  (active ≥ {MIN_USER_INTERACTIONS} interactions)')
print(f'  Edges (train_uo)  : {len(train_uo_filtered):,}')
print(f'')
print(f'  item_embeddings   : {item_embeddings_arr.shape}    → {ITEM_EMB_FILE}')
print(f'  user_embeddings   : {user_embeddings_arr.shape}  → {USER_EMB_FILE}')
print(f'  outfit_embeddings : {outfit_embeddings_arr.shape} → {OUTFIT_EMB_FILE}')
print(f'')
print('✅ Notebook 1 hoàn thành! Sẵn sàng cho Notebook 2 (xây graph & matrices).')

=== TỔNG KẾT Notebook 1 ===
  MIN_USER_INTERACTIONS : 4

  Items sau filter  : 14,419  (có ảnh, thuộc outfit)
  Outfits sau filter: 6,622
  Users sau filter  : 127,536  (active ≥ 4 interactions)
  Edges (train_uo)  : 127,536

  item_embeddings   : (14419, 65)    → /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/embeddings/item_embeddings.npy
  user_embeddings   : (25263, 65)  → /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/embeddings/user_embeddings.npy
  outfit_embeddings : (6622, 65) → /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/embeddings/outfit_embeddings.npy

✅ Notebook 1 hoàn thành! Sẵn sàng cho Notebook 2 (xây graph & matrices).
